Cross comparison of the AI model with other datasets


In [2]:
%pip install ultralytics kagglehub opencv-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
from torch.utils.data import Dataset, Subset, DataLoader
import torch
import torchvision.transforms as T
from pathlib import Path
from io import BytesIO
from ultralytics import YOLO
import numpy as np
from functools import lru_cache
import kagglehub
import cv2
import logging

/home/willm277/Desktop/AIDetectionCrossValidation/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Plant Village import
# path = kagglehub.dataset_download("mohitsingh1804/plantvillage", output_dir="datasets/plant_village", force_download=True)

# PlantDoc import
path = kagglehub.dataset_download(
    "andresmgs/plantdec", output_dir="datasets/plant_doc", force_download=True)

100%|██████████| 818M/818M [01:30<00:00, 9.50MB/s] 

Extracting files...


100%|██████████| 74.1M/74.1M [00:07<00:00, 10.0MB/s]

Extracting files...


In [25]:
# Model and basic function definition

potatoModel = YOLO("weights/potatoYOLOv11.pt")
tomatoModel = YOLO("weights/tomatoYOLOv11.pt")
onionModel = YOLO("weights/onionYOLOv11.pt")
lettuceModel = YOLO("weights/lettuceDisease.pt")
appleModel = YOLO("weights/appleDisease.pt")
strawberryModel = YOLO("weights/strawberryDisease.pt")


def predictBoxes(image, model):

    results = model(image)
    boxes = results[0].boxes
    output = []
    for box in boxes:
        x_min, y_min, x_max, y_max = box.xyxy[0].tolist()
        confidence = box.conf[0].item()
        class_id = int(box.cls[0].item())
        label = results[0].names[class_id]
        output.append({
            "label": label,
            "confidence": confidence,
            "box": [x_min, y_min, x_max, y_max]
        })
    return output


def get_label_tomato(image):
    result = predictBoxes(image, tomatoModel)
    return result


def get_label_potato(image):
    result = predictBoxes(image, potatoModel)
    return result


def get_label_onion(image):
    result = predictBoxes(image, onionModel)
    return result


def get_label_lettuce(image):
    result = predictBoxes(image, lettuceModel)
    return result


def get_label_apple(image):
    result = predictBoxes(image, appleModel)
    return result


def get_label_strawberry(image):
    result = predictBoxes(image, strawberryModel)
    return result

In [ ]:
# PlantDoc
print("init")

# Dataset sub-datasets and Data Loader

# Full dataset containing all images from PlantDoc (including non detected diseases)

# Dataset to filter (Doesnt load images as it would be too much storage)


class PlantDoc_valid_labels_Dataset(Dataset):
    def __init__(self, dataset_path):
        self.ds_path: Path = Path(dataset_path)
        self.image_paths = self.get_all_paths()

    def __getitem__(self, idx):
        # Path chosen through indexing
        path_image: Path = self.image_paths[idx]
        # Creating the path for the labels
        path_parts = list(path_image.parts)
        path_parts[-2] = "labels"
        labels_path_wo_suffix = Path(*path_parts)
        path_labels = labels_path_wo_suffix.with_suffix(".txt")

        # Getting the labels from .txt file
        labels = []
        with open(path_labels, "r", encoding="utf-8") as file:
            for line in file:
                label, x1, y1, x2, y2 = map(str.strip, line.split(" "))
                labels.append({"label": label, "box": (x1, y1, x2, y2)})
        return (path_image, labels)

    def __len__(self):
        # Returns the length of the dataset
        return len(self.image_paths)

    def get_all_paths(self):
        # get all paths for the images in the dataset (labels paths are inferred later)
        image_paths = list(self.ds_path.rglob("**/*.jpg"))
        return image_paths


# Define which labels will be tested
plantDoc_labels = {"plants":
                   {
                       "Potato":
                       {
                           "labels": {"11": "EarlyBlight", "12": "LateBlight", "13": "Healthy"},
                           "idx": {"11", "12", "13"},
                       },
                       "Tomato":
                       {
                           "labels": {"19": "EarlyBlight", "20": "SeptoriaLeafSpot", "21": "BacterialSpot", "22": "TomatoLateBlight", "23": "MosaicVirus", "24": "YellowLeafCurlVirus", "25": "HealthyTomato", "26": "LeafMold", "27": "SpiderMites"},
                           # index
                           "idx": {"19", "20", "21", "22", "23", "24", "25", "26", "27"}
                       },
                   },
                   "used_idx": {"11", "12", "13", "19", "20", "21", "22", "23", "24", "25", "26", "27"}
                   }

# Checks if the label is valid


def is_valid(item, all_labels):
    _, item_labels = item
    for item_label in item_labels:
        if item_label["label"] not in all_labels["used_idx"]:
            return False
    if not item_labels:
        return False

    return True


 # Sorts items
print("loading labels ds...")
labels_dataset = PlantDoc_valid_labels_Dataset("datasets/plant_doc")
print("labels ds finished. filtering items...")
filtered_items = [
    item for item in labels_dataset if is_valid(item, plantDoc_labels)]

print("filtering complete.")
# Filtered Dataset with labels


class PlantDoc_Dataset_distilled(Dataset):
    def __init__(self, items, labels_dict):
        self.resize_transform = T.Resize(size=(256, 256), antialias=True)
        self.max_labels = 16  # cuz i need uniform tensor sizes rip
        self.max_plants = 1
        self.items = items
        self.labels_dict = labels_dict

    def __getitem__(self, idx):
        img_path, original_labels = self.items[idx]
        labels = [item.copy() for item in original_labels]
        plants = set()
        for label in labels:
            for plant, dicts in self.labels_dict["plants"].items():
                if label["label"] in dicts["idx"]:
                    plants.add(plant)

        with open(img_path, "rb") as f:
            img_buffer = BytesIO(f.read())
        raw_bytes = np.frombuffer(img_buffer.getbuffer(), dtype=np.uint8)
        img_array_np = cv2.imdecode(raw_bytes, cv2.IMREAD_COLOR)[
            :, :, ::-1]  # reverts the colors into rgb
        img_tensor = torch.from_numpy(
            np.ascontiguousarray(img_array_np)).permute(2, 0, 1)
        img = self.resize_transform(img_tensor)

        plant_list = list(plants)
        if not plant_list:
            print("NO PLANT FOUND WTF")
            raise ("piss")
        plant_list.extend(["None",]*(self.max_plants-len(plant_list)))

        padding_dict = {"label": "None", "box": ("0", "0", "0", "0")}
        labels.extend([padding_dict] * (self.max_labels - len(labels)))
        labels = labels[:self.max_labels]

        return plant_list, img, labels

    def __len__(self):
        return len(self.items)


print("Creating new filtered dataset")
filtered_plantdoc_dataset = PlantDoc_Dataset_distilled(
    filtered_items, plantDoc_labels)

print("Creating data loader")
# loader = DataLoader(filtered_plantdoc_dataset, batch_size= 4, prefetch_factor=2, pin_memory=True, num_workers=2)
loader = DataLoader(filtered_plantdoc_dataset, batch_size=4,)

# Convert dataloader to an iterator and get the first batch

print(" Getting batches")
for batch in loader:
    plants, images, labels = batch
    print(plants, labels)

init
loading labels ds...
labels ds finished. filtering items...
filtering complete.
Creating new filtered dataset
Creating data loader
 Getting batches
[('Tomato', 'Tomato', 'Tomato', 'Potato')] [{'label': ['26', '23', '19', '13'], 'box': [('0.7822916666666665', '0.28515625', '0.46888888888888886', '0.6843417553191489'), ('0.43281250000000004', '0.6967213114754098', '0.5325925925925926', '0.738'), ('0.3590277777777778', '0.4921875', '0.23555555555555555', '0.12998670212765956'), ('0.25625000000000003', '0.4988290398126463', '0.5111111111111112', '0.40299999999999997')]}, {'label': ['26', '23', '19', '13'], 'box': [('0.5364583333333333', '0.51015625', '0.6077777777777778', '0.5374002659574468'), ('0.7864583333333334', '0.572599531615925', '0.29333333333333333', '0.59175'), ('0.6493055555555555', '0.6890625000000001', '0.4044444444444444', '0.2576462765957447'), ('0.415625', '0.8173302107728337', '0.5837037037037037', '0.4174999999999999')]}, {'label': ['26', 'None', '19', '13'], 'box':

In [ ]:
[('Tomato', 'Potato', 'Tomato', 'None'), ('None', 'None', 'None', 'None')]
[{'label': ['22', '12', '23', 'None'],
    'box': [('0.6791666666666667', '0.5066666666666667', '0.5', '0'), ('0.2730496453900709', '0.5', '0.4972972972972973', '0'), ('0.4783333333333334', '0.98', '0.9766666666666667', '0'), ('0.4231678486997636\n', '0.9898477157360406', '0.9708108108108108', '0')]}, {'label': ['22', 'None', 'None', 'None'], 'box': [('0.4033333333333333', '0', '0', '0'), ('0.40425531914893614', '0', '0', '0'), ('0.75', '0', '0', '0'), ('0.5059101654846335', '0', '0', '0')]}, {'label': ['None', 'None', 'None', 'None'], 'box': [('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0')]}, {
 'label': ['None', 'None', 'None', 'None'],
    'box': [('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0')]}, {'label': ['None', 'None', 'None', 'None'], 'box': [('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0')]}, {'label': ['None', 'None', 'None', 'None'], 'box': [('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0')]}, {'label': ['None', 'None', 'None', 'None'], 'box': [('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0')]}, {'label': ['None', 'None', 'None', 'None'], 'box': [('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0'), ('0', '0', '0', '0')]}]